# Multi-Vendor Backend Ranking

Choosing the right quantum backend is one of the highest-impact decisions in any experiment. The wrong choice can waste your budget on noisy results or overpay for fidelity you don't need.

`BackendRecommender` answers the question: **which backend should I use for this circuit today?**

This notebook covers:

1. Setting up `BackendRecommender` and adding backends
2. Analyzing a circuit with `recommender.analyze()`
3. Reading `RecommendationReport` and `BackendAnalysis` fields
4. Comparing IBM vs Rigetti vs IonQ vs IQM
5. Fidelity-per-dollar analysis
6. Circuit-specific recommendations (small vs large circuits)
7. Exploring `BACKEND_CONFIGS`
8. Building a recommendation dashboard

## 1. Setup and Imports

In [1]:
import math

from qiskit import QuantumCircuit

from qb_compiler import BACKEND_CONFIGS, BackendSpec
from qb_compiler.recommender import (
    BackendAnalysis,
    BackendRecommender,
    RecommendationReport,
)

## 2. Exploring BACKEND_CONFIGS

Before recommending backends, let's survey what's available. `BACKEND_CONFIGS` contains hardware specifications for every supported backend.

In [2]:
print(f"{'Backend':20s}  {'Provider':12s}  {'Qubits':>8s}  {'CX Error':>10s}  {'RO Error':>10s}  {'$/shot':>10s}")
print("-" * 78)

for name, spec in sorted(BACKEND_CONFIGS.items(), key=lambda x: x[1].provider):
    print(
        f"{name:20s}  {spec.provider:12s}  "
        f"{spec.n_qubits:>8d}  "
        f"{spec.median_cx_error:>10.4f}  "
        f"{spec.median_readout_error:>10.4f}  "
        f"${spec.cost_per_shot:>9.5f}"
    )

Backend               Provider        Qubits    CX Error    RO Error      $/shot
------------------------------------------------------------------------------
ibm_fez               ibm                156      0.0050      0.0100  $  0.00016
ibm_torino            ibm                133      0.0060      0.0120  $  0.00014
ibm_marrakesh         ibm                156      0.0055      0.0110  $  0.00016
ionq_aria             ionq                25      0.0040      0.0030  $  0.30000
ionq_forte            ionq                36      0.0030      0.0030  $  0.30000
iqm_garnet            iqm                 20      0.0150      0.0200  $  0.00045
iqm_emerald           iqm                  5      0.0080      0.0150  $  0.00020
quantinuum_h2         quantinuum          32      0.0010      0.0020  $  8.00000
rigetti_ankaa         rigetti             84      0.0200      0.0300  $  0.00035


In [3]:
# Inspect a single backend spec in detail
backend_name = "ibm_fez"
spec = BACKEND_CONFIGS[backend_name]

print(f"BackendSpec for {backend_name}:")
print(f"  Provider:            {spec.provider}")
print(f"  Physical qubits:     {spec.n_qubits}")
print(f"  Median CX error:     {spec.median_cx_error:.4f}")
print(f"  Median readout err:  {spec.median_readout_error:.4f}")
print(f"  Cost per shot:       ${spec.cost_per_shot:.5f}")
print(f"  Basis gates:         {spec.basis_gates}")

BackendSpec for ibm_fez:
  Provider:            ibm
  Physical qubits:     156
  Median CX error:     0.0050
  Median readout err:  0.0100
  Cost per shot:       $0.00016
  Basis gates:         ('id', 'rz', 'sx', 'x', 'cz', 'reset')


## 3. Setting Up BackendRecommender

Create a recommender and register the backends you want to compare. The recommender loads calibration data automatically for known backends.

In [4]:
recommender = BackendRecommender(n_seeds=10, shots=4096)

# Add backends for comparison
recommender.add_backend("ibm_fez")
recommender.add_backend("ibm_torino")
recommender.add_backend("rigetti_ankaa")
recommender.add_backend("ionq_aria")
recommender.add_backend("iqm_garnet")

print("Recommender configured with 5 backends.")

Recommender configured with 5 backends.


`add_backend()` returns `self`, so you can chain calls:
```python
rec = BackendRecommender().add_backend("ibm_fez").add_backend("ibm_torino")
```

## 4. Analyzing a Circuit

Call `recommender.analyze(circuit)` to get a ranked recommendation across all registered backends.

In [5]:
# Build a GHZ-5 circuit
ghz5 = QuantumCircuit(5, 5, name="GHZ-5")
ghz5.h(0)
for i in range(4):
    ghz5.cx(i, i + 1)
ghz5.measure(range(5), range(5))

report = recommender.analyze(ghz5)

# The report has a rich __str__ representation
print(report)

/home/dministrator/miniconda3/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Failed to analyze ionq_aria: No target or calibration for ionq_aria


Failed to analyze iqm_garnet: No target or calibration for iqm_garnet


Backend Recommendation Report: GHZ-5 (5q)

-----------------------------------------------------------------------------------
       Backend |  Est.Fid | 2Q Gates |  Depth |      Status |  Cost/4096 |    Fid/$
-----------------------------------------------------------------------------------
    ibm_torino |   0.9030 |        4 |      6 |      VIABLE |    $0.5734 |      1.6 <-- BEST
       ibm_fez |   0.8621 |        4 |     10 |      VIABLE |    $0.6554 |      1.3
 rigetti_ankaa |   0.7969 |        4 |     10 |      VIABLE |    $1.4336 |      0.6
-----------------------------------------------------------------------------------

Recommendation: ibm_torino (fidelity=0.903)

Analysis time: 1899ms


## 5. RecommendationReport Fields

The report contains the overall recommendation plus per-backend analysis results.

In [6]:
print(f"Circuit:              {report.circuit_name}")
print(f"Qubits:               {report.n_qubits}")
print(f"Best fidelity:        {report.best_fidelity}")
print(f"Best value ($/fid):   {report.best_value}")
print(f"Analysis time:        {report.total_analysis_time_ms:.0f} ms")
print(f"Backends analyzed:    {len(report.analyses)}")
print()
print(f"Recommendation:       {report.recommendation}")
print()
if report.warnings:
    print("Warnings:")
    for w in report.warnings:
        print(f"  - {w}")

Circuit:              GHZ-5
Qubits:               5
Best fidelity:        ibm_torino
Best value ($/fid):   ibm_torino
Analysis time:        1899 ms
Backends analyzed:    3

Recommendation:       Recommendation: ibm_torino (fidelity=0.903)

Warnings:
  - ionq_aria: analysis failed (No target or calibration for ionq_aria)
  - iqm_garnet: analysis failed (No target or calibration for iqm_garnet)


## 6. BackendAnalysis Fields

Each `BackendAnalysis` object contains detailed metrics for one backend.

In [7]:
# Pick the top-ranked analysis
best = report.analyses[0]

print(f"=== {best.backend} ===")
print(f"  provider:             {best.provider}")
print(f"  estimated_fidelity:   {best.estimated_fidelity:.4f}")
print(f"  two_qubit_gate_count: {best.two_qubit_gate_count}")
print(f"  depth:                {best.depth}")
print(f"  viable:               {best.viable}")
print(f"  status:               {best.status}")
print(f"  cost_per_4096_shots:  ${best.cost_per_4096_shots:.4f}")
print(f"  fidelity_per_dollar:  {best.fidelity_per_dollar:.1f}")
print(f"  best_seed:            {best.best_seed}")
print(f"  physical_qubits:      {best.physical_qubits}")
print(f"  viable_depth:         {best.viable_depth}")
print(f"  analysis_time_ms:     {best.analysis_time_ms:.1f}")

=== ibm_torino ===
  provider:             ibm
  estimated_fidelity:   0.9030
  two_qubit_gate_count: 4
  depth:                6
  viable:               True
  status:               VIABLE
  cost_per_4096_shots:  $0.5734
  fidelity_per_dollar:  1.6
  best_seed:            0
  physical_qubits:      [4, 3, 2, 1, 0]
  viable_depth:         450
  analysis_time_ms:     391.0


## 7. Comparing All Backends

Let's build a detailed comparison table.

In [8]:
print(f"{'Backend':>15s} | {'Provider':>10s} | {'Fidelity':>10s} | {'2Q Gates':>10s} | "
      f"{'Depth':>6s} | {'Status':>11s} | {'Cost/4096':>10s} | {'Fid/$':>8s}")
print("-" * 100)

for a in report.analyses:
    cost_str = f"${a.cost_per_4096_shots:.4f}" if a.cost_per_4096_shots else "N/A"
    fpd_str = f"{a.fidelity_per_dollar:.1f}" if a.fidelity_per_dollar else "N/A"
    marker = ""
    if a.backend == report.best_fidelity:
        marker = " <-- BEST"
    elif a.backend == report.best_value:
        marker = " <-- VALUE"
    print(
        f"{a.backend:>15s} | {a.provider:>10s} | {a.estimated_fidelity:>10.4f} | "
        f"{a.two_qubit_gate_count:>10d} | {a.depth:>6d} | {a.status:>11s} | "
        f"{cost_str:>10s} | {fpd_str:>8s}{marker}"
    )

        Backend |   Provider |   Fidelity |   2Q Gates |  Depth |      Status |  Cost/4096 |    Fid/$
----------------------------------------------------------------------------------------------------
     ibm_torino |        ibm |     0.9030 |          4 |      6 |      VIABLE |    $0.5734 |      1.6 <-- BEST
        ibm_fez |        ibm |     0.8621 |          4 |     10 |      VIABLE |    $0.6554 |      1.3
  rigetti_ankaa |    rigetti |     0.7969 |          4 |     10 |      VIABLE |    $1.4336 |      0.6


## 8. Fidelity-Per-Dollar Analysis

Raw fidelity isn't always the right metric. If a backend gives 5% more fidelity but costs 100x more, the cheap backend wins on value.

The `fidelity_per_dollar` field captures this: higher is better.

In [9]:
# Sort by fidelity-per-dollar (best value first)
viable = [a for a in report.analyses if a.viable and a.fidelity_per_dollar is not None]
by_value = sorted(viable, key=lambda a: -(a.fidelity_per_dollar or 0))

print("Backends ranked by value (fidelity per dollar):")
print(f"{'Rank':>6s}  {'Backend':>15s}  {'Fidelity':>10s}  {'Cost':>10s}  {'Fid/$':>10s}")
print("-" * 58)

for i, a in enumerate(by_value, 1):
    print(
        f"{i:>6d}  {a.backend:>15s}  {a.estimated_fidelity:>10.4f}  "
        f"${a.cost_per_4096_shots:>8.4f}  {a.fidelity_per_dollar:>10.1f}"
    )

Backends ranked by value (fidelity per dollar):
  Rank          Backend    Fidelity        Cost       Fid/$
----------------------------------------------------------
     1       ibm_torino      0.9030  $  0.5734         1.6
     2          ibm_fez      0.8621  $  0.6554         1.3
     3    rigetti_ankaa      0.7969  $  1.4336         0.6


In [10]:
# Compare: best fidelity vs best value
if report.best_fidelity and report.best_value and report.best_fidelity != report.best_value:
    best_f = next(a for a in report.analyses if a.backend == report.best_fidelity)
    best_v = next(a for a in report.analyses if a.backend == report.best_value)

    fid_diff = best_f.estimated_fidelity - best_v.estimated_fidelity
    cost_ratio = (best_f.cost_per_4096_shots or 0) / (best_v.cost_per_4096_shots or 1)

    print(f"Best fidelity:  {best_f.backend} (fidelity={best_f.estimated_fidelity:.4f}, cost=${best_f.cost_per_4096_shots:.4f})")
    print(f"Best value:     {best_v.backend} (fidelity={best_v.estimated_fidelity:.4f}, cost=${best_v.cost_per_4096_shots:.4f})")
    print(f"")
    print(f"Fidelity difference: {fid_diff:+.4f}")
    print(f"Cost ratio:          {cost_ratio:.1f}x")
    print(f"")
    if fid_diff < 0.05 and cost_ratio > 5:
        print(f"The value pick ({best_v.backend}) is likely the better choice: similar")
        print(f"fidelity at a fraction of the cost.")
    else:
        print(f"The fidelity pick ({best_f.backend}) provides meaningfully better results.")
else:
    print(f"Best fidelity and best value are the same backend: {report.best_fidelity}")

Best fidelity and best value are the same backend: ibm_torino


## 9. Circuit-Specific Recommendations

Different circuits favor different backends. Small circuits with few 2Q gates favor backends with low per-gate error (IonQ, Quantinuum). Large circuits with many qubits and gates favor backends with many qubits and low per-shot cost (IBM).

In [11]:
# Small circuit: 2-qubit Bell state
bell = QuantumCircuit(2, 2, name="Bell-2")
bell.h(0)
bell.cx(0, 1)
bell.measure([0, 1], [0, 1])

report_small = recommender.analyze(bell)

print(f"=== Small circuit ({report_small.circuit_name}) ===")
print(f"Best fidelity: {report_small.best_fidelity}")
print(f"Best value:    {report_small.best_value}")
for a in report_small.analyses:
    print(f"  {a.backend:>15s}  fidelity={a.estimated_fidelity:.4f}  2Q={a.two_qubit_gate_count}")

Failed to analyze ionq_aria: No target or calibration for ionq_aria


Failed to analyze iqm_garnet: No target or calibration for iqm_garnet


=== Small circuit (Bell-2) ===
Best fidelity: ibm_torino
Best value:    ibm_torino
       ibm_torino  fidelity=0.9720  2Q=1
          ibm_fez  fidelity=0.9430  2Q=1
    rigetti_ankaa  fidelity=0.9286  2Q=1


In [12]:
# Large circuit: 20-qubit GHZ
ghz20 = QuantumCircuit(20, 20, name="GHZ-20")
ghz20.h(0)
for i in range(19):
    ghz20.cx(i, i + 1)
ghz20.measure(range(20), range(20))

report_large = recommender.analyze(ghz20)

print(f"=== Large circuit ({report_large.circuit_name}) ===")
print(f"Best fidelity: {report_large.best_fidelity}")
print(f"Best value:    {report_large.best_value}")
for a in report_large.analyses:
    print(f"  {a.backend:>15s}  fidelity={a.estimated_fidelity:.4f}  status={a.status}")

Failed to analyze ibm_torino: 'A connected component of the DAGCircuit is too large for any of the connected components in the coupling map.'


Failed to analyze rigetti_ankaa: 'A connected component of the DAGCircuit is too large for any of the connected components in the coupling map.'


Failed to analyze ionq_aria: No target or calibration for ionq_aria


Failed to analyze iqm_garnet: No target or calibration for iqm_garnet


=== Large circuit (GHZ-20) ===
Best fidelity: ibm_fez
Best value:    ibm_fez
          ibm_fez  fidelity=0.6097  status=VIABLE


In [13]:
# QAOA circuit: moderate depth, ring connectivity
def make_qaoa(n: int, p: int) -> QuantumCircuit:
    qc = QuantumCircuit(n, n, name=f"QAOA-{n}-p{p}")
    for q in range(n):
        qc.h(q)
    for _ in range(p):
        for i in range(n):
            j = (i + 1) % n
            qc.cx(i, j)
            qc.rz(0.5, j)
            qc.cx(i, j)
        for i in range(n):
            qc.rx(0.7, i)
    qc.measure(range(n), range(n))
    return qc


qaoa8 = make_qaoa(8, p=3)
report_qaoa = recommender.analyze(qaoa8)

print(f"=== QAOA circuit ({report_qaoa.circuit_name}) ===")
print(f"Best fidelity: {report_qaoa.best_fidelity}")
print(f"Best value:    {report_qaoa.best_value}")
for a in report_qaoa.analyses:
    cost_str = f"${a.cost_per_4096_shots:.4f}" if a.cost_per_4096_shots else "N/A"
    print(f"  {a.backend:>15s}  fidelity={a.estimated_fidelity:.4f}  cost={cost_str}  {a.status}")

Failed to analyze ibm_torino: 'A connected component of the DAGCircuit is too large for any of the connected components in the coupling map.'


Failed to analyze rigetti_ankaa: 'A connected component of the DAGCircuit is too large for any of the connected components in the coupling map.'


Failed to analyze ionq_aria: No target or calibration for ionq_aria


Failed to analyze iqm_garnet: No target or calibration for iqm_garnet


=== QAOA circuit (QAOA-8-p3) ===
Best fidelity: ibm_fez
Best value:    ibm_fez
          ibm_fez  fidelity=0.6541  cost=$0.6554  VIABLE


## 10. Scaling Analysis: How Recommendations Change with Circuit Size

Let's see how the best backend changes as we scale up the qubit count.

In [14]:
print(f"{'N Qubits':>10s}  {'Best Fidelity':>18s}  {'Best Value':>18s}  {'Top Fidelity':>14s}")
print("-" * 65)

for n in [2, 4, 6, 8, 10, 15, 20]:
    ghz = QuantumCircuit(n, n, name=f"GHZ-{n}")
    ghz.h(0)
    for i in range(n - 1):
        ghz.cx(i, i + 1)
    ghz.measure(range(n), range(n))

    r = recommender.analyze(ghz)
    top_fid = r.analyses[0].estimated_fidelity if r.analyses else 0
    print(
        f"{n:>10d}  "
        f"{r.best_fidelity or 'none':>18s}  "
        f"{r.best_value or 'none':>18s}  "
        f"{top_fid:>14.4f}"
    )

  N Qubits       Best Fidelity          Best Value    Top Fidelity
-----------------------------------------------------------------


Failed to analyze ionq_aria: No target or calibration for ionq_aria


Failed to analyze iqm_garnet: No target or calibration for iqm_garnet


         2          ibm_torino          ibm_torino          0.9720


Failed to analyze ionq_aria: No target or calibration for ionq_aria


...

Failed to analyze ibm_torino: 'A connected component of the DAGCircuit is too large for any of the connected components in the coupling map.'


Failed to analyze rigetti_ankaa: 'A connected component of the DAGCircuit is too large for any of the connected components in the coupling map.'


Failed to analyze ionq_aria: No target or calibration for ionq_aria


Failed to analyze iqm_garnet: No target or calibration for iqm_garnet


        20             ibm_fez             ibm_fez          0.6097


## 11. Physical Qubit Selection

The recommender also tells you *which* physical qubits were selected by the transpiler. This is useful for understanding why one backend outperforms another.

In [15]:
ghz5 = QuantumCircuit(5, 5, name="GHZ-5")
ghz5.h(0)
for i in range(4):
    ghz5.cx(i, i + 1)
ghz5.measure(range(5), range(5))

report = recommender.analyze(ghz5)

print("Physical qubit selection per backend:")
for a in report.analyses:
    print(f"  {a.backend:>15s}  qubits={a.physical_qubits}  seed={a.best_seed}")

Failed to analyze ionq_aria: No target or calibration for ionq_aria


Failed to analyze iqm_garnet: No target or calibration for iqm_garnet


Physical qubit selection per backend:
       ibm_torino  qubits=[4, 3, 2, 1, 0]  seed=0
          ibm_fez  qubits=[132, 131, 138, 151, 152]  seed=0
    rigetti_ankaa  qubits=[4, 3, 2, 1, 0]  seed=0


## 12. Building a Recommendation Dashboard

Let's build a function that produces a complete dashboard comparing multiple circuits across multiple backends.

In [16]:
def recommendation_dashboard(
    circuits: list[QuantumCircuit],
    recommender: BackendRecommender,
) -> None:
    """Print a dashboard comparing circuits across backends."""
    print("=" * 80)
    print("BACKEND RECOMMENDATION DASHBOARD")
    print("=" * 80)

    for circ in circuits:
        report = recommender.analyze(circ)
        print(f"\n--- {report.circuit_name} ({report.n_qubits}q) ---")
        print(f"{'Backend':>15s}  {'Status':>11s}  {'Fidelity':>10s}  {'Cost':>10s}  {'Fid/$':>8s}  {'Note':>10s}")
        print("-" * 72)

        for a in report.analyses:
            cost_str = f"${a.cost_per_4096_shots:.4f}" if a.cost_per_4096_shots else "N/A"
            fpd_str = f"{a.fidelity_per_dollar:.0f}" if a.fidelity_per_dollar else "N/A"
            note = ""
            if a.backend == report.best_fidelity:
                note = "BEST"
            elif a.backend == report.best_value:
                note = "VALUE"
            print(
                f"{a.backend:>15s}  {a.status:>11s}  "
                f"{a.estimated_fidelity:>10.4f}  {cost_str:>10s}  "
                f"{fpd_str:>8s}  {note:>10s}"
            )

        if report.warnings:
            for w in report.warnings:
                if "NOT VIABLE" not in w:  # skip per-backend not-viable warnings
                    print(f"  WARNING: {w}")


print("Dashboard function defined.")

Dashboard function defined.


In [17]:
# Run the dashboard with a variety of circuits
dashboard_circuits = [
    # Simple
    (lambda: QuantumCircuit(2, 2, name="Bell"))(),
    # Medium GHZ
    (lambda n=8: (qc := QuantumCircuit(n, n, name=f"GHZ-{n}"),
                  qc.h(0),
                  [qc.cx(i, i+1) for i in range(n-1)],
                  qc.measure(range(n), range(n)),
                  qc)[-1])(),
    # QAOA
    make_qaoa(6, p=2),
]

# Fix the Bell circuit (needs gates)
dashboard_circuits[0].h(0)
dashboard_circuits[0].cx(0, 1)
dashboard_circuits[0].measure([0, 1], [0, 1])

recommendation_dashboard(dashboard_circuits, recommender)

BACKEND RECOMMENDATION DASHBOARD


Failed to analyze ionq_aria: No target or calibration for ionq_aria


Failed to analyze iqm_garnet: No target or calibration for iqm_garnet



--- Bell (2q) ---
        Backend       Status    Fidelity        Cost     Fid/$        Note
------------------------------------------------------------------------
     ibm_torino       VIABLE      0.9720     $0.5734         2        BEST
        ibm_fez       VIABLE      0.9430     $0.6554         1            
  rigetti_ankaa       VIABLE      0.9286     $1.4336         1            


Failed to analyze ibm_torino: 'A connected component of the DAGCircuit is too large for any of the connected components in the coupling map.'


...

Failed to analyze iqm_garnet: No target or calibration for iqm_garnet



--- GHZ-8 (8q) ---
        Backend       Status    Fidelity        Cost     Fid/$        Note
------------------------------------------------------------------------
        ibm_fez       VIABLE      0.8604     $0.6554         1        BEST


Failed to analyze ionq_aria: No target or calibration for ionq_aria


Failed to analyze iqm_garnet: No target or calibration for iqm_garnet



--- QAOA-6-p2 (6q) ---
        Backend       Status    Fidelity        Cost     Fid/$        Note
------------------------------------------------------------------------
        ibm_fez       VIABLE      0.8000     $0.6554         1        BEST
     ibm_torino       VIABLE      0.7012     $0.5734         1            
  rigetti_ankaa       VIABLE      0.3407     $1.4336         0            


## 13. Handling Warnings

The recommender generates warnings when backends can't produce useful results. These are important signals.

In [18]:
# Build a circuit that's too deep for some backends
deep = QuantumCircuit(4, 4, name="DeepCircuit")
for _ in range(100):
    for i in range(3):
        deep.cx(i, i + 1)
    for i in range(4):
        deep.rz(0.5, i)
deep.measure(range(4), range(4))

report_deep = recommender.analyze(deep)

print(f"Circuit: {report_deep.circuit_name}")
print(f"Best fidelity: {report_deep.best_fidelity}")
print()

if report_deep.warnings:
    print("Warnings:")
    for w in report_deep.warnings:
        print(f"  - {w}")

Failed to analyze ionq_aria: No target or calibration for ionq_aria


Failed to analyze iqm_garnet: No target or calibration for iqm_garnet


Circuit: DeepCircuit
Best fidelity: ibm_fez

Warnings:
  - ionq_aria: analysis failed (No target or calibration for ionq_aria)
  - iqm_garnet: analysis failed (No target or calibration for iqm_garnet)
  - rigetti_ankaa: fidelity 0.0031 is below noise floor: results will be random noise.


## Summary

In this notebook you learned:

- **`BackendRecommender`**: register backends with `add_backend()`, analyze circuits with `analyze()`
- **`RecommendationReport`**: `best_fidelity`, `best_value`, `analyses`, `warnings`, and the one-line `recommendation` property
- **`BackendAnalysis`**: per-backend metrics: `estimated_fidelity`, `cost_per_4096_shots`, `fidelity_per_dollar`, `physical_qubits`, `viable_depth`, `status`
- **Fidelity-per-dollar**: the right metric when you're budget-constrained
- **Circuit-specific recommendations**: small circuits favor low-error backends (IonQ), large circuits favor high-qubit-count backends (IBM)
- **`BACKEND_CONFIGS`**: hardware specs for every supported backend
- **Dashboard pattern**: comparing multiple circuits across multiple backends in a single view

Next: [04: Selective Dynamical Decoupling](04_dynamical_decoupling.ipynb)